In [ ]:
# ============================================================
# Exercise 3: Cadence + Aktivitaetsklassifikation (einfach)
# ============================================================

from pathlib import Path
import numpy as np
from scipy.signal import butter, filtfilt, welch

# kleiner Workaround, weil DataProcessor intern Excersice_3 importiert
try:
    from DataProcessor import DataProcessor
except ModuleNotFoundError:
    import sys
    import types
    sys.modules["Excersice_3"] = types.ModuleType("Excersice_3")
    from DataProcessor import DataProcessor


# 1 = resting, 2 = walking, 3 = running, 0 = unknown
ACTIVITY_NAMES = {0: "Unknown", 1: "Ruhen", 2: "Gehen", 3: "Rennen"}


def total_acc_magnitude(x, y, z):
    mag = np.sqrt(x**2 + y**2 + z**2)
    mag = mag - np.mean(mag)
    return mag


def bandpass_filter(signal, fs, lowcut=0.5, highcut=5.0, order=4):
    nyquist = fs / 2.0
    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype="band")
    filtered = filtfilt(b, a, signal)
    return filtered


def estimate_cadence(block, fs):
    corr = np.correlate(block, block, mode="full")
    corr = corr[len(corr) // 2 :]
    
    min_lag = int(fs / 3.5)
    max_lag = int(fs / 0.8)
    
    if max_lag >= len(corr):
        max_lag = len(corr) - 1
    if min_lag >= max_lag:
        return 0.0
    
    part = corr[min_lag:max_lag]
    if len(part) == 0:
        return 0.0
    
    peak_lag = np.argmax(part) + min_lag
    cadence = 60.0 / (peak_lag / fs)
    return cadence


def dominant_frequency(signal, fs):
    # bleibt drin wie gewuenscht
    sig_filt = bandpass_filter(signal, fs)
    nperseg = min(2 * fs, len(sig_filt))
    freqs, pxx = welch(sig_filt, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
    mask = (freqs >= 0.5) & (freqs <= 3.5)
    
    if not np.any(mask):
        return 0.0
    
    idx = np.argmax(pxx[mask])
    return freqs[mask][idx]


def classify_activity(cadence):
    # Ausgabe wie in der Aufgabe: 0..3
    if cadence <= 0:
        return 0
    elif cadence < 70:
        return 1
    elif cadence < 130:
        return 2
    else:
        return 3


def cadence_activity_algorithm(accelerationData, fs):
    """
    Geforderte Hauptfunktion der Aufgabe.
    Input: accelerationData als Nx3 oder 3xN, fs in Hz
    Output: cadence_vector, activity_vector (pro 5 Sekunden)
    """
    arr = np.array(accelerationData)
    
    # sowohl Nx3 als auch 3xN erlauben
    if arr.ndim != 2:
        return [], []
    if arr.shape[1] == 3:
        acc = arr
    elif arr.shape[0] == 3:
        acc = arr.T
    else:
        return [], []

    x = acc[:, 0]
    y = acc[:, 1]
    z = acc[:, 2]

    block_samples = int(5 * fs)
    num_blocks = len(x) // block_samples

    cadence_vector = []
    activity_vector = []

    for i in range(num_blocks):
        start = i * block_samples
        end = start + block_samples

        mag = total_acc_magnitude(x[start:end], y[start:end], z[start:end])
        mag = bandpass_filter(mag, fs)

        cad = estimate_cadence(mag, fs)
        _dom = dominant_frequency(mag, fs)  # nur zur Vollstaendigkeit
        act = classify_activity(cad)

        cadence_vector.append(float(cad))
        activity_vector.append(int(act))

    return cadence_vector, activity_vector


def analyze_file(file_path):
    dp = DataProcessor("rawdata/X22/")
    dp.loadRawData(str(file_path))

    devices = dp.getDevices()
    if len(devices) == 0:
        return 0, [], []

    dp.loadRawDataDevice(devices[0])
    x = dp.dfAcc["x"].values
    y = dp.dfAcc["y"].values
    z = dp.dfAcc["z"].values
    fs = dp.fs

    accelerationData = np.column_stack((x, y, z))
    cadence_vector, activity_vector = cadence_activity_algorithm(accelerationData, fs)
    return fs, cadence_vector, activity_vector


# ------------------------------------------------------------
# Aufgabe 3: Daten laden und analysieren
# ------------------------------------------------------------
exercise3_files = [
    Path("rawdata/X22/Excersice_2/rennen_hakim/Mes_1/Hakim_rennnen_5.pickle"),
    Path("rawdata/X22/schnell_Laufen10.pickle"),
    Path("rawdata/X22/rennen1.pickle"),
]

for file_path in exercise3_files:
    print(f"\n--- {file_path} ---")

    if not file_path.exists():
        print("Datei nicht gefunden.")
        continue

    fs, cadences, activities = analyze_file(file_path)

    print(f"Samplingrate: {fs} Hz")
    print(f"Anzahl 5s-Bloecke: {len(cadences)}")
    if len(cadences) > 0:
        print(f"Cadence (steps/min): {[f'{c:.1f}' for c in cadences]}")
        print(f"Aktivitaeten: {[(a, ACTIVITY_NAMES[a]) for a in activities]}")
        print(f"Mittlere Cadence: {np.mean(cadences):.1f} steps/min")
    else:
        print("Keine vollstaendigen Bloecke.")


--- rawdata\X22\normal_gehen3.pickle ---
Samplingrate: 200 Hz
Anzahl 5s-Bloecke: 7
Cadence (steps/min): ['67.4', '68.6', '70.2', '70.2', '70.2', '64.9', '210.5']
Aktivitaeten: [(1, 'Ruhen'), (1, 'Ruhen'), (2, 'Gehen'), (2, 'Gehen'), (2, 'Gehen'), (1, 'Ruhen'), (3, 'Rennen')]
Mittlere Cadence: 88.8 steps/min

--- rawdata\X22\schnell_Laufen10.pickle ---
Samplingrate: 200 Hz
Anzahl 5s-Bloecke: 13
Cadence (steps/min): ['73.6', '74.1', '73.6', '72.3', '69.4', '68.2', '69.0', '69.0', '68.2', '70.6', '72.3', '210.5', '210.5']
Aktivitaeten: [(2, 'Gehen'), (2, 'Gehen'), (2, 'Gehen'), (2, 'Gehen'), (1, 'Ruhen'), (1, 'Ruhen'), (1, 'Ruhen'), (1, 'Ruhen'), (1, 'Ruhen'), (2, 'Gehen'), (2, 'Gehen'), (3, 'Rennen'), (3, 'Rennen')]
Mittlere Cadence: 92.4 steps/min

--- rawdata\X22\rennen1.pickle ---
Samplingrate: 200 Hz
Anzahl 5s-Bloecke: 10
Cadence (steps/min): ['99.2', '179.1', '65.2', '83.3', '131.9', '210.5', '210.5', '48.2', '57.7', '210.5']
Aktivitaeten: [(2, 'Gehen'), (3, 'Rennen'), (1, 'Ruhen')

In [8]:
# ============================================================
# Exercise 4: Test des Algorithmus auf gelabelten Ordnerdaten
# ============================================================

from pathlib import Path
import numpy as np


def infer_ground_truth_label(file_path):
    text = str(file_path).lower().replace("\\", "/")
    name = file_path.name.lower()

    # Ground Truth auch als 0..3
    if "rennen" in text:
        return 3
    elif "schnell_gehen" in text or "laufen_el" in text or "schnell_laufen" in text:
        return 3
    elif "normal_gehen" in text or name.startswith("ex2_gehen"):
        return 2
    else:
        return 0


dataset_root = Path("rawdata/X22/Excersice_2")
all_files = sorted(dataset_root.rglob("*.pickle"))

eval_files = []
for f in all_files:
    eval_files.append(f)

print(f"Gefundene Testdateien: {len(eval_files)}")

y_true = []
y_pred = []
per_file = []

for file_path in eval_files:
    true_label = infer_ground_truth_label(file_path)
    fs, cadence_vector, pred_blocks = analyze_file(file_path)

    if len(pred_blocks) == 0:
        continue

    for p in pred_blocks:
        y_true.append(true_label)
        y_pred.append(p)

    file_acc = np.mean(np.array(pred_blocks) == true_label)
    per_file.append((file_path, true_label, len(pred_blocks), float(file_acc)))

print(f"Ausgewertete Bloecke gesamt: {len(y_true)}")
print(f"Dateien mit mindestens 1 Block: {len(per_file)}")

if len(y_true) == 0:
    print("Keine auswertbaren Bloecke gefunden.")
else:
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    accuracy = np.mean(y_true_arr == y_pred_arr)

    labels = [0, 1, 2, 3]
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for t, p in zip(y_true, y_pred):
        i = labels.index(t)
        j = labels.index(p)
        cm[i, j] += 1

    print(f"\nAccuracy (blockbasiert): {accuracy:.3f}")
    print("Labels:", labels, "->", [ACTIVITY_NAMES[l] for l in labels])
    print("\nConfusion Matrix (Zeilen=true, Spalten=pred):")
    print(cm)

    print("\nEinfache Klassen-Recall-Werte:")
    for i, label in enumerate(labels):
        total_true = np.sum(cm[i, :])
        if total_true == 0:
            recall = 0.0
        else:
            recall = cm[i, i] / total_true
        print(f"{label} ({ACTIVITY_NAMES[label]}): Recall={recall:.3f}")

    print("\nDateiuebersicht (erste 15):")
    for file_path, gt_label, n_blocks, file_acc in per_file[:15]:
        print(
            f"{file_path} | GT={gt_label} ({ACTIVITY_NAMES[gt_label]}) | "
            f"Bloecke={n_blocks} | Datei-Accuracy={file_acc:.3f}"
        )

Gefundene Testdateien: 64
Ausgewertete Bloecke gesamt: 455
Dateien mit mindestens 1 Block: 64

Accuracy (blockbasiert): 0.288
Labels: [0, 1, 2, 3] -> ['Unknown', 'Ruhen', 'Gehen', 'Rennen']

Confusion Matrix (Zeilen=true, Spalten=pred):
[[  0   0   0   0]
 [  0   0   0   0]
 [  0  65  59  43]
 [  0 112 104  72]]

Einfache Klassen-Recall-Werte:
0 (Unknown): Recall=0.000
1 (Ruhen): Recall=0.000
2 (Gehen): Recall=0.353
3 (Rennen): Recall=0.250

Dateiuebersicht (erste 15):
rawdata\X22\Excersice_2\EX2_gehen1.pickle | GT=2 (Gehen) | Bloecke=13 | Datei-Accuracy=0.462
rawdata\X22\Excersice_2\EX2_gehen2.pickle | GT=2 (Gehen) | Bloecke=10 | Datei-Accuracy=0.200
rawdata\X22\Excersice_2\EX2_gehen3.pickle | GT=2 (Gehen) | Bloecke=1 | Datei-Accuracy=0.000
rawdata\X22\Excersice_2\EX2_gehen4.pickle | GT=2 (Gehen) | Bloecke=7 | Datei-Accuracy=0.429
rawdata\X22\Excersice_2\EX2_gehen5.pickle | GT=2 (Gehen) | Bloecke=5 | Datei-Accuracy=0.400
rawdata\X22\Excersice_2\normal_gehen_hakim\Mes_1\19700101_000004